# Notebook 10b — Level 3, 7 epoch (số bước cập nhật ngang L1)

**Câu hỏi:** mục 8 của báo cáo cho rằng các lỗi Ineffective "tự tin cao" cần thông tin ở mức toàn bài
(counterclaim không được phản bác, position trơ trọi…). Cho mô hình đọc cả bài thì có tốt hơn L1 không?

**Kiến trúc:** toàn văn bài (`full_text`) được chèn token đánh dấu quanh mỗi đơn vị có nhãn, theo loại:
`[CLAIM] … [/CLAIM]`, `[EVIDENCE] … [/EVIDENCE]`, … DeBERTa-v3-base đọc cả bài (tối đa 1.536 token; bài dài hơn
được chia cửa sổ chồng 256 token), biểu diễn mỗi đơn vị = trung bình hidden state từ token mở đến token đóng,
rồi một lớp tuyến tính phân 3 nhãn. Một lượt forward cho mọi đơn vị trong bài.

**Giữ như L0/L1:** deberta-v3-base, lr 2e-5, 3 epoch, warmup 0.1, AdamW, class weight balanced (trên đơn vị),
chọn epoch theo Macro-F1 validation, cùng split. Batch hiệu dụng: 8 bài (~70 đơn vị) mỗi bước.
Chỉ validation; so với L1/L0 tuần 11 trên cùng 29.102 đơn vị, bootstrap theo bài. **Không chấm official test.**


**Vì sao có bản 7 epoch?** Ở notebook 10 (3 epoch), Macro-F1 validation còn tăng đều (0,521 → 0,551 → 0,576): mô hình chưa hội tụ.
Lý do: mỗi bước cập nhật của L3 gồm 8 bài (~74 đơn vị), nên 3 epoch chỉ có ~4.700 bước, so với ~10.800 bước của L1 (batch 32 đơn vị).
Bản này train 7 epoch (~10.900 bước, ngang L1); mọi thứ khác giữ nguyên, epoch tốt nhất vẫn chọn theo Macro-F1 validation.

In [ ]:
# ===== 1. MÔI TRƯỜNG =====
# transformers 5.x làm DeBERTa-v3 train hỏng (đã kiểm chứng tuần 11) -> bắt buộc 4.x.
# Trên Colab CLI: `colab install transformers==4.57.6` rồi `colab restart-kernel` trước khi chạy notebook.
import importlib.metadata
TV = importlib.metadata.version("transformers")
assert TV.startswith("4."), f"Cần transformers 4.x, đang có {TV}"

import os, gc, json, time, random, hashlib, math, zipfile, copy
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import requests
import torch
import torch.nn as nn
import transformers
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, cohen_kappa_score, f1_score,
                             precision_score, recall_score)
from sklearn.utils.class_weight import compute_class_weight

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
QUICK = os.environ.get("QUICK", "0") == "1"      # chạy thử: vài chục bài, 1 epoch — KHÔNG dùng số này
GPU = torch.cuda.get_device_name(0)
USE_BF16 = torch.cuda.is_bf16_supported()
print(f"transformers {TV} | torch {torch.__version__} | {GPU} | bf16={USE_BF16} | QUICK={QUICK}")
print(f"VRAM {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB")

transformers 4.57.6 | torch 2.11.0+cu130 | NVIDIA L4 | bf16=True | QUICK=False
VRAM 22.0 GB


In [ ]:
KEY = "L3_7ep"

In [ ]:
# ===== 2. DỮ LIỆU + SPLIT =====
WORK = "/content"
DATA_DIR = f"{WORK}/data"
OUT_DIR = f"{WORK}/out_{KEY}" + ("_QUICK" if QUICK else "")
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)
TRAIN_CSV = f"{DATA_DIR}/persuade_corpus_2.0_train.csv"
TRAIN_URL = "https://drive.usercontent.google.com/download?id=13phHyDzIsb0MHyJr6q-B-qIa9P2tM135&export=download&confirm=t"
TRAIN_SHA256 = "f61319edd8bf16a982711ea0399fad59c05afaec05cdf0767f16a2c05c467e23"
LABEL2ID = {"Ineffective": 0, "Adequate": 1, "Effective": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}


def log(msg):
    line = f"[{datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')} UTC] [{KEY}] {msg}"
    print(line, flush=True)
    with open(f"{OUT_DIR}/run_log.txt", "a", encoding="utf-8") as f:
        f.write(line + "\n")


def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 22), b""):
            h.update(block)
    return h.hexdigest()


if not os.path.exists(TRAIN_CSV):
    t0 = time.time()
    with requests.get(TRAIN_URL, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(TRAIN_CSV + ".part", "wb") as f:
            for block in r.iter_content(chunk_size=1 << 22):
                f.write(block)
    os.replace(TRAIN_CSV + ".part", TRAIN_CSV)
    log(f"Tải train CSV xong sau {time.time() - t0:.0f}s")
assert sha256_of(TRAIN_CSV) == TRAIN_SHA256, "SHA-256 của train CSV không khớp"


def load_units():
    # Cùng quy tắc lọc với notebook 07 -> row_idx trùng với val_predictions_*.csv tuần 11
    cols = ["essay_id_comp", "full_text", "discourse_start", "discourse_end", "discourse_text",
            "discourse_type", "discourse_effectiveness", "prompt_name"]
    df = pd.read_csv(TRAIN_CSV, usecols=cols, dtype=str, low_memory=False)
    df = df.dropna(subset=["discourse_effectiveness", "discourse_text"]).copy()
    for c in ["discourse_text", "discourse_type", "essay_id_comp"]:
        df[c] = df[c].astype(str).str.strip()
    df = df[df["discourse_text"].str.len() > 0].reset_index(drop=True)
    df["row_idx"] = np.arange(len(df))
    df["prompt_name"] = df["prompt_name"].fillna("").str.replace('"', "", regex=False).str.strip()
    df["label"] = df["discourse_effectiveness"].map(LABEL2ID).astype(int)
    df["start"] = pd.to_numeric(df["discourse_start"], errors="coerce").fillna(0).astype(int)
    df["end"] = pd.to_numeric(df["discourse_end"], errors="coerce").fillna(0).astype(int)
    # Đơn vị trước/sau trong cùng bài (theo essay_id_comp, sắp theo vị trí) — như notebook 07
    s = df.sort_values(["essay_id_comp", "start", "row_idx"], kind="mergesort")
    g = s.groupby("essay_id_comp")["discourse_text"]
    for k in (1, 2):
        s[f"prev{k}"] = g.shift(k).fillna("")
        s[f"next{k}"] = g.shift(-k).fillna("")
    return s.sort_values("row_idx").reset_index(drop=True)


units = load_units()
assert len(units) == 144289, len(units)
REF = pd.read_csv(f"{WORK}/val_predictions_L1.csv")          # dự đoán L1 tuần 11 trên validation
REF_L0 = pd.read_csv(f"{WORK}/val_predictions_L0.csv")
units["split"] = np.where(units["row_idx"].isin(set(REF["row_idx"])), "val", "train")
assert units.groupby("essay_id_comp")["split"].nunique().max() == 1, "Một bài nằm ở cả train và val"
train_df = units[units["split"] == "train"].reset_index(drop=True)
val_df = units[units["split"] == "val"].reset_index(drop=True)
assert len(train_df) == 115187 and len(val_df) == 29102
assert (val_df["row_idx"].values == REF["row_idx"].values).all()
assert (val_df["label"].values == REF["true_label"].values).all()
log(f"Train {len(train_df):,} đơn vị / {train_df.essay_id_comp.nunique():,} bài | "
    f"Val {len(val_df):,} đơn vị / {val_df.essay_id_comp.nunique():,} bài (khớp split tuần 9–11)")

FULL_VAL = val_df  # giữ bản đầy đủ để so sánh
if QUICK:
    rng = np.random.RandomState(0)
    pick = lambda d, k: d[d.essay_id_comp.isin(rng.choice(d.essay_id_comp.unique(), k, replace=False))]
    train_df = pick(train_df, 80).reset_index(drop=True)
    val_df = pick(val_df, 30).reset_index(drop=True)
    log(f"QUICK: train {len(train_df)} / val {len(val_df)} đơn vị")

[2026-10-31 05:41:59 UTC] [L3_7ep] Train 115,187 đơn vị / 12,475 bài | Val 29,102 đơn vị / 3,119 bài (khớp split tuần 9–11)


In [ ]:
# ===== 3. HÀM ĐÁNH GIÁ =====
def metrics_dict(y, pred):
    f1c = f1_score(y, pred, labels=[0, 1, 2], average=None, zero_division=0)
    return {"macro_f1": f1_score(y, pred, labels=[0, 1, 2], average="macro", zero_division=0),
            "accuracy": accuracy_score(y, pred), "balanced_accuracy": balanced_accuracy_score(y, pred),
            "qwk": cohen_kappa_score(y, pred, weights="quadratic"),
            "f1_ineffective": f1c[0], "f1_adequate": f1c[1], "f1_effective": f1c[2],
            "recall_ineffective": recall_score(y == 0, pred == 0, zero_division=0),
            "precision_ineffective": precision_score(y == 0, pred == 0, zero_division=0)}


def macro(y, p):
    return f1_score(y, p, labels=[0, 1, 2], average="macro", zero_division=0)


def paired_bootstrap(essays, y, base, new, n_boot=2000, seed=42):
    """Δ Macro-F1 (new − base), lấy mẫu lại theo bài."""
    codes = pd.Series(essays).astype("category").cat.codes.values
    members = [np.flatnonzero(codes == k) for k in range(codes.max() + 1)]
    rng = np.random.default_rng(seed)
    d = []
    for _ in range(n_boot):
        idx = np.concatenate([members[k] for k in rng.integers(0, len(members), len(members))])
        d.append(macro(y[idx], new[idx]) - macro(y[idx], base[idx]))
    d = np.array(d)
    return {"delta": round(macro(y, new) - macro(y, base), 4), "ci_low": round(np.percentile(d, 2.5), 4),
            "ci_high": round(np.percentile(d, 97.5), 4), "p_one_sided_le0": round(float((d <= 0).mean()), 4),
            "n_units": len(y), "n_essays": len(members), "n_boot": n_boot}


def save_predictions(probs, name):
    out = val_df[["row_idx", "essay_id_comp", "discourse_type", "prompt_name"]].copy()
    out["true_label"] = val_df["label"].values
    out["pred_label"] = probs.argmax(1)
    for c in range(3):
        out[f"pred_proba_{c}"] = np.round(probs[:, c], 6)
    path = f"{OUT_DIR}/val_predictions_{name}.csv"
    out.to_csv(path, index=False)
    log(f"Lưu {path}")
    return out


def compare_with_reference(pred, name, subsets=None):
    """So với L0 / L1 tuần 11 trên CÙNG đơn vị validation, bootstrap theo bài."""
    ref1 = REF.set_index("row_idx").loc[val_df["row_idx"], "pred_label"].values
    ref0 = REF_L0.set_index("row_idx").loc[val_df["row_idx"], "pred_label"].values
    y, ess = val_df["label"].values, val_df["essay_id_comp"].values
    subsets = subsets or {"all": np.ones(len(y), bool)}
    rows = []
    for sname, m in subsets.items():
        if m.sum() == 0:
            continue
        for bname, base in [("L1 (tuần 11)", ref1), ("L0 (tuần 11)", ref0)]:
            r = paired_bootstrap(ess[m], y[m], base[m], pred[m], n_boot=200 if QUICK else 2000)
            rows.append({"model": name, "subset": sname, "baseline": bname,
                         "macro_f1_model": round(macro(y[m], pred[m]), 4),
                         "macro_f1_baseline": round(macro(y[m], base[m]), 4), **r})
    df = pd.DataFrame(rows)
    df.to_csv(f"{OUT_DIR}/compare_{name}_vs_L0_L1.csv", index=False)
    print(df.to_string(index=False))
    return df

In [ ]:
# ===== 3b. DỰNG DỮ LIỆU LEVEL 3 (cả bài + token đánh dấu) =====
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup, set_seed

CFG = dict(model_name="microsoft/deberta-v3-base", max_len=1536, overlap=256, eff_essays=8, lr=2e-5,
           weight_decay=0.01, epochs=7, warmup_ratio=0.1, seed=42, dropout=0.1)
if QUICK:
    CFG.update(epochs=1)
TYPES = sorted(units["discourse_type"].unique())
TAG = {t: t.upper().replace(" ", "_") for t in TYPES}
S_TOK = {t: f"[{TAG[t]}]" for t in TYPES}
E_TOK = {t: f"[/{TAG[t]}]" for t in TYPES}
tok = AutoTokenizer.from_pretrained(CFG["model_name"], use_fast=True)
tok.add_special_tokens({"additional_special_tokens": list(S_TOK.values()) + list(E_TOK.values())})
S_IDS = set(tok.convert_tokens_to_ids(list(S_TOK.values())))
E_IDS = set(tok.convert_tokens_to_ids(list(E_TOK.values())))
log(f"Token đánh dấu: {list(S_TOK.values())}")


def essay_items(g):
    g = g.sort_values(["start", "row_idx"], kind="mergesort")
    ft = g["full_text"].iloc[0]
    parts, pos = [], 0
    for r in g.itertuples():
        s = max(r.start, pos); e = max(r.end, s)
        parts += [ft[pos:s], f" {S_TOK[r.discourse_type]} ", ft[s:e], f" {E_TOK[r.discourse_type]} "]
        pos = e
    parts.append(ft[pos:])
    ids = tok("".join(parts), add_special_tokens=False)["input_ids"]
    starts = [i for i, x in enumerate(ids) if x in S_IDS]
    ends = [i for i, x in enumerate(ids) if x in E_IDS]
    assert len(starts) == len(ends) == len(g), (len(starts), len(ends), len(g))
    W = CFG["max_len"] - 2
    wins, a = [], 0
    while True:
        b = min(a + W, len(ids)); wins.append((a, b))
        if b >= len(ids):
            break
        a += W - CFG["overlap"]
    per_win = [[] for _ in wins]
    n_partial = 0
    for (s, e), lab, rid in zip(zip(starts, ends), g["label"].values, g["row_idx"].values):
        wi = next((i for i, (a, b) in enumerate(wins) if a <= s and e < b), None)
        if wi is None:                       # đơn vị dài hơn phần chồng: lấy cửa sổ chứa token mở, cắt ở cuối
            wi = next(i for i, (a, b) in enumerate(wins) if a <= s < b); n_partial += 1
        a, b = wins[wi]
        per_win[wi].append((s - a + 1, min(e, b - 1) - a + 1, int(lab), int(rid)))   # +1 cho [CLS]
    items = []
    for (a, b), us in zip(wins, per_win):
        if us:
            items.append({"ids": [tok.cls_token_id] + ids[a:b] + [tok.sep_token_id],
                          "spans": [(u[0], u[1]) for u in us], "labels": [u[2] for u in us],
                          "row_idx": [u[3] for u in us]})
    return items, len(ids), len(wins), n_partial


def build(d, name):
    items, lens, nwin, npart = [], [], [], 0
    for _, g in d.groupby("essay_id_comp", sort=False):
        it, L, w, p = essay_items(g)
        items += it; lens.append(L); nwin.append(w); npart += p
    lens = np.array(lens)
    log(f"[{name}] {len(lens):,} bài -> {len(items):,} cửa sổ | token/bài p50/p90/p99/max = "
        f"{np.percentile(lens, [50, 90, 99, 100]).round().tolist()} | bài > {CFG['max_len']} token: "
        f"{(lens > CFG['max_len'] - 2).mean():.2%} | đơn vị bị cắt cuối: {npart}")
    assert sum(len(x["labels"]) for x in items) == len(d)
    return items


t0 = time.time()
train_items, val_items = build(train_df, "train"), build(val_df, "val")
log(f"Dựng dữ liệu xong sau {time.time() - t0:.0f}s")
ex = val_items[0]
print(tok.decode(ex["ids"][:220]))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


/usr/local/lib/python3.13/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


[2026-10-31 05:42:12 UTC] [L3_7ep] Token đánh dấu: ['[CLAIM]', '[CONCLUDING_STATEMENT]', '[COUNTERCLAIM]', '[EVIDENCE]', '[LEAD]', '[POSITION]', '[REBUTTAL]']


[2026-10-31 05:42:54 UTC] [L3_7ep] [train] 12,475 bài -> 12,481 cửa sổ | token/bài p50/p90/p99/max = [470.0, 819.0, 1201.0, 1800.0] | bài > 1536 token: 0.07% | đơn vị bị cắt cuối: 0


[2026-10-31 05:43:04 UTC] [L3_7ep] [val] 3,119 bài -> 3,119 cửa sổ | token/bài p50/p90/p99/max = [464.0, 811.0, 1204.0, 1515.0] | bài > 1536 token: 0.00% | đơn vị bị cắt cuối: 0


[2026-10-31 05:43:04 UTC] [L3_7ep] Dựng dữ liệu xong sau 52s


[CLS][LEAD] People are debating whether if drivers should or should not use cell phones while operating a vehicle. Some people think that using a cell phone while operating a vehicle should be allowed because they try multi-tasking and have important calls while some are addicted and can't stay off their phone. Other people says that you shouldn't be able to be on your cell phone while operating a vehicle because the cell phone can cause accidents while driving.[/LEAD][POSITION] I also think that you shouldn't use your phone while operating a vehicle[/POSITION][CLAIM] because it can cause vehicle collisions[/CLAIM] ,[CLAIM] slow reaction time[/CLAIM] ,[CLAIM] and fatal injuries[/CLAIM] . Therefore, driving can cause many accidents that can be fatal to the driver and passengers if there is any and cell phones should only be used when not operating a vehicle.[CLAIM] The first reason why the use of cell phones shouldn't be allowed while operating a vehicle because there are high chances o

In [ ]:
# ===== 4. MÔ HÌNH + TRAIN LEVEL 3 =====
class EssayUnitClassifier(nn.Module):
    def __init__(self, name, vocab_size):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(name)
        self.backbone.resize_token_embeddings(vocab_size)
        self.drop = nn.Dropout(CFG["dropout"])
        self.head = nn.Linear(self.backbone.config.hidden_size, 3)

    def forward(self, input_ids, attention_mask, span_b, span_s, span_e):
        h = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        T = h.size(1)
        ar = torch.arange(T, device=h.device)
        w = ((ar[None, :] >= span_s[:, None]) & (ar[None, :] <= span_e[:, None])).to(h.dtype)
        w = w / w.sum(1, keepdim=True)
        pooled = torch.einsum("ut,uth->uh", w, h[span_b])          # trung bình hidden state trong đơn vị
        return self.head(self.drop(pooled)).float()


def collate(batch):
    T = max(len(x["ids"]) for x in batch)
    ids = torch.full((len(batch), T), tok.pad_token_id, dtype=torch.long)
    att = torch.zeros((len(batch), T), dtype=torch.long)
    sb, ss, se, lab, rid = [], [], [], [], []
    for i, x in enumerate(batch):
        ids[i, :len(x["ids"])] = torch.tensor(x["ids"]); att[i, :len(x["ids"])] = 1
        for (s, e), l, r in zip(x["spans"], x["labels"], x["row_idx"]):
            sb.append(i); ss.append(s); se.append(e); lab.append(l); rid.append(r)
    t = lambda v: torch.tensor(v, dtype=torch.long)
    return ids, att, t(sb), t(ss), t(se), t(lab), np.array(rid)


def make_batches(items, bs, shuffle, rng):
    idx = np.arange(len(items))
    if shuffle:
        rng.shuffle(idx)
        mega = bs * 50                                  # gom bài dài gần nhau -> bớt padding
        idx = np.concatenate([sorted(idx[i:i + mega], key=lambda j: len(items[j]["ids"]))
                              for i in range(0, len(idx), mega)])
        bl = [idx[i:i + bs] for i in range(0, len(idx), bs)]
        rng.shuffle(bl)
    else:
        idx = sorted(idx, key=lambda j: len(items[j]["ids"]))
        bl = [idx[i:i + bs] for i in range(0, len(idx), bs)]
    return bl


def seed_all(seed):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed); set_seed(seed)


DTYPE = torch.bfloat16 if USE_BF16 else torch.float16


def pick_batch(model):
    total = torch.cuda.get_device_properties(0).total_memory
    fake = {"ids": [5] * CFG["max_len"], "spans": [(i * 40 + 1, i * 40 + 30) for i in range(20)],
            "labels": [1] * 20, "row_idx": list(range(20))}
    for bs in (8, 4, 2, 1):
        try:
            torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
            ids, att, sb, ss, se, lab, _ = collate([fake] * bs)
            with torch.autocast("cuda", dtype=DTYPE):
                model(ids.cuda(), att.cuda(), sb.cuda(), ss.cuda(), se.cuda()).sum().backward()
            model.zero_grad(set_to_none=True)
            peak = torch.cuda.max_memory_allocated()
            log(f"Thử {bs} bài x {CFG['max_len']} token: đỉnh VRAM {peak / 2**30:.1f}/{total / 2**30:.1f} GB")
            if peak < 0.80 * total:
                return bs, max(1, CFG["eff_essays"] // bs)
        except torch.cuda.OutOfMemoryError:
            log(f"{bs} bài tràn VRAM")
        model.zero_grad(set_to_none=True); torch.cuda.empty_cache()
    return 1, CFG["eff_essays"]


@torch.no_grad()
def predict(model, items, bs):
    model.eval()
    out_r, out_p = [], []
    for b in make_batches(items, bs * 2, False, None):
        ids, att, sb, ss, se, lab, rid = collate([items[j] for j in b])
        with torch.autocast("cuda", dtype=DTYPE):
            logits = model(ids.cuda(), att.cuda(), sb.cuda(), ss.cuda(), se.cuda())
        out_r.append(rid); out_p.append(torch.softmax(logits, -1).cpu().numpy())
    model.train()
    r, p = np.concatenate(out_r), np.concatenate(out_p)
    pos = pd.Series(np.arange(len(r)), index=r)
    order = pos.loc[val_df["row_idx"].values].values         # về đúng thứ tự val_df
    return p[order]


seed_all(CFG["seed"])
model = EssayUnitClassifier(CFG["model_name"], len(tok)).cuda()
per_dev, accum = pick_batch(model)
weights = compute_class_weight("balanced", classes=np.array([0, 1, 2]), y=train_df["label"].values)
loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float).cuda())
no_decay = ["bias", "LayerNorm.weight"]
params = [{"params": [p for n, p in model.named_parameters() if not any(k in n for k in no_decay)],
           "weight_decay": CFG["weight_decay"]},
          {"params": [p for n, p in model.named_parameters() if any(k in n for k in no_decay)], "weight_decay": 0.0}]
opt = torch.optim.AdamW(params, lr=CFG["lr"])
rng = np.random.default_rng(CFG["seed"])
n_batches = math.ceil(len(train_items) / per_dev)
total_steps = math.ceil(n_batches / accum) * CFG["epochs"]
sched = get_linear_schedule_with_warmup(opt, int(CFG["warmup_ratio"] * total_steps), total_steps)
log(f"batch {per_dev} bài x grad_accum {accum} | {n_batches} batch/epoch | {total_steps} bước tối ưu | "
    f"class weights {np.round(weights, 4).tolist()}")

best, best_state, history = -1, None, []
t_all = time.time()
for ep in range(1, CFG["epochs"] + 1):
    t0, run_loss, step = time.time(), 0.0, 0
    for bi, b in enumerate(make_batches(train_items, per_dev, True, rng)):
        ids, att, sb, ss, se, lab, _ = collate([train_items[j] for j in b])
        with torch.autocast("cuda", dtype=DTYPE):
            logits = model(ids.cuda(), att.cuda(), sb.cuda(), ss.cuda(), se.cuda())
        loss = loss_fn(logits, lab.cuda()) / accum
        loss.backward()
        run_loss += loss.item() * accum
        if (bi + 1) % accum == 0 or bi + 1 == n_batches:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad(set_to_none=True); step += 1
            if step % (10 if QUICK else 200) == 0:
                log(f"epoch {ep} bước {step} | loss {run_loss / (bi + 1):.4f} | {time.time() - t0:.0f}s")
    assert np.isfinite(run_loss), "Loss NaN"
    probs = predict(model, val_items, per_dev)
    m = metrics_dict(val_df["label"].values, probs.argmax(1))
    history.append({"epoch": ep, "train_loss": run_loss / n_batches, **{k: float(v) for k, v in m.items()},
                    "epoch_minutes": (time.time() - t0) / 60})
    log(f"epoch {ep}: train loss {run_loss / n_batches:.4f} | val Macro-F1 {m['macro_f1']:.4f} | "
        f"F1 Ineff {m['f1_ineffective']:.4f} | Acc {m['accuracy']:.4f} | {(time.time() - t0) / 60:.1f} phút")
    if m["macro_f1"] > best:
        best, best_probs = m["macro_f1"], probs
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
minutes = (time.time() - t_all) / 60
log(f"Train xong sau {minutes:.1f} phút | best val Macro-F1 {best:.4f}")
json.dump({"cfg": CFG, "batch_essays": per_dev, "grad_accum": accum, "history": history, "best_metric": best,
           "train_minutes": round(minutes, 1), "gpu": GPU, "transformers": TV, "class_weights": weights.tolist(),
           "quick": QUICK}, open(f"{OUT_DIR}/train_history_{KEY}.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
PROBS = best_probs
save_predictions(PROBS, KEY)
CMP = compare_with_reference(PROBS.argmax(1), KEY)
# Theo loại đơn vị (giả thuyết: Counterclaim/Rebuttal/Position cần thông tin toàn bài)
ref1 = REF.set_index("row_idx").loc[val_df["row_idx"], "pred_label"].values
y, pred = val_df["label"].values, PROBS.argmax(1)
rows = []
for t in sorted(val_df.discourse_type.unique()):
    mm = val_df.discourse_type.values == t
    rows.append({"discourse_type": t, "n": int(mm.sum()), "L3": round(macro(y[mm], pred[mm]), 4),
                 "L1": round(macro(y[mm], ref1[mm]), 4), "delta": round(macro(y[mm], pred[mm]) - macro(y[mm], ref1[mm]), 4)})
by_type = pd.DataFrame(rows); by_type.to_csv(f"{OUT_DIR}/L3_vs_L1_by_type.csv", index=False)
print(by_type.to_string(index=False))
del model, opt; gc.collect(); torch.cuda.empty_cache()

[2026-10-31 05:43:11 UTC] [L3_7ep] 8 bài tràn VRAM


[2026-10-31 05:43:14 UTC] [L3_7ep] Thử 4 bài x 1536 token: đỉnh VRAM 14.0/22.0 GB


[2026-10-31 05:43:14 UTC] [L3_7ep] batch 4 bài x grad_accum 2 | 3121 batch/epoch | 10927 bước tối ưu | class weights [7.4787, 0.425, 1.9484]


[2026-10-31 05:44:53 UTC] [L3_7ep] epoch 1 bước 200 | loss 1.0540 | 99s


[2026-10-31 05:46:41 UTC] [L3_7ep] epoch 1 bước 400 | loss 0.9759 | 207s


...

[2026-10-31 07:31:01 UTC] [L3_7ep] epoch 7 bước 1400 | loss 0.2726 | 806s


[2026-10-31 07:33:24 UTC] [L3_7ep] epoch 7: train loss 0.2746 | val Macro-F1 0.5637 | F1 Ineff 0.3719 | Acc 0.7278 | 15.8 phút


[2026-10-31 07:33:24 UTC] [L3_7ep] Train xong sau 110.2 phút | best val Macro-F1 0.5698


[2026-10-31 07:33:25 UTC] [L3_7ep] Lưu /content/out_L3_7ep/val_predictions_L3_7ep.csv


 model subset     baseline  macro_f1_model  macro_f1_baseline   delta  ci_low  ci_high  p_one_sided_le0  n_units  n_essays  n_boot
L3_7ep    all L1 (tuần 11)          0.5698             0.5809 -0.0112 -0.0204  -0.0024           0.9945    29102      3119    2000
L3_7ep    all L0 (tuần 11)          0.5698             0.5723 -0.0025 -0.0124   0.0069           0.6915    29102      3119    2000
      discourse_type     n     L3     L1   delta
               Claim 10135 0.5220 0.5277 -0.0057
Concluding Statement  2711 0.5518 0.5916 -0.0398
        Counterclaim  1148 0.4817 0.4988 -0.0171
            Evidence  9249 0.6111 0.6328 -0.0217
                Lead  1902 0.5444 0.5585 -0.0141
            Position  3089 0.5709 0.5297  0.0412
            Rebuttal   868 0.5154 0.5145  0.0009


In [ ]:
# ===== XUẤT KẾT QUẢ =====
m = metrics_dict(val_df["label"].values, PROBS.argmax(1))
summary = {"model": KEY, **{k: round(float(v), 4) for k, v in m.items()}, "n_val_units": len(val_df),
           "quick": QUICK, "gpu": GPU, "transformers": TV, "finished_at": datetime.now(timezone.utc).isoformat()}
json.dump(summary, open(f"{OUT_DIR}/summary_{KEY}.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
log(f"KẾT QUẢ {KEY} (validation): " + " | ".join(f"{k} {v}" for k, v in summary.items() if isinstance(v, float)))
bundle = f"{WORK}/results_{KEY}" + ("_QUICK" if QUICK else "") + ".zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as zf:
    for fn in sorted(os.listdir(OUT_DIR)):
        zf.write(f"{OUT_DIR}/{fn}", f"{os.path.basename(OUT_DIR)}/{fn}")
print("Đã đóng gói:", bundle, os.path.getsize(bundle), "bytes")

[2026-10-31 07:33:55 UTC] [L3_7ep] KẾT QUẢ L3_7ep (validation): macro_f1 0.5698 | accuracy 0.7271 | balanced_accuracy 0.615 | qwk 0.4223 | f1_ineffective 0.3857 | f1_adequate 0.8153 | f1_effective 0.5084 | recall_ineffective 0.43 | precision_ineffective 0.3497


Đã đóng gói: /content/results_L3_7ep.zip 510799 bytes
